# Threshold Analysis for Token-Level Self-Repair (Google Colab)

This notebook performs comprehensive threshold analysis to understand how different uncertainty thresholds affect:
- Accuracy before and after repair
- Repair trigger rates and success rates
- Uncertainty calibration
- Performance overhead
- Per-dataset performance

## Setup for Google Colab


In [ ]:
# Step 1: Install dependencies
%pip install -q torch transformers accelerate bitsandbytes huggingface_hub
%pip install -q numpy pandas matplotlib seaborn tqdm scipy pydantic rich langgraph langchain-core

# Step 2: Clone or download the repository
# Option A: If your repo is on GitHub, uncomment and use:
# !git clone https://github.com/yourusername/Agentic_LLM.git
# %cd Agentic_LLM

# Option B: If you need to upload files manually, upload the 'src' folder to Colab
# Then set the path below to where you uploaded it

import sys
from pathlib import Path
import os

# Set up paths for Colab
if 'google.colab' in str(get_ipython()):
    # Running in Colab
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Option 1: If you uploaded the project to Google Drive
    PROJECT_PATH = Path('/content/drive/MyDrive/Agentic_LLM')
    
    # Option 2: If you cloned from GitHub (uncomment if using Option A above)
    # PROJECT_PATH = Path('/content/Agentic_LLM')
    
    # Option 3: If you want to work in current directory, upload 'src' folder here
    # PROJECT_PATH = Path('/content')
    
    if not PROJECT_PATH.exists():
        print(f"⚠️ Project path {PROJECT_PATH} not found!")
        print("Please either:")
        print("1. Upload your project to Google Drive at the path above, OR")
        print("2. Clone from GitHub (uncomment the git clone line above), OR")
        print("3. Upload the 'src' folder to /content/ and set PROJECT_PATH = Path('/content')")
        raise FileNotFoundError(f"Project not found at {PROJECT_PATH}")
    
    sys.path.insert(0, str(PROJECT_PATH))
    os.chdir(PROJECT_PATH)
    print(f"✅ Project loaded from: {PROJECT_PATH}")
else:
    # Running locally
    PROJECT_PATH = Path().resolve().parent
    sys.path.insert(0, str(PROJECT_PATH))
    print(f"✅ Running locally from: {PROJECT_PATH}")

# Verify imports work
try:
    from src.token_self_repair.llm import load_llama
    from src.token_self_repair.pipelines import default_reasoning_coordinator
    from src.token_self_repair.evaluation import ReasoningEvaluationRunner
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Make sure the 'src' folder is in the project path.")


In [ ]:
import json
import time
from datetime import datetime
from typing import Dict, List, Optional
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.token_self_repair.llm import load_llama
from src.token_self_repair.pipelines import default_reasoning_coordinator
from src.token_self_repair.evaluation import ReasoningEvaluationRunner
from src.token_self_repair.config import ProjectConfig, Thresholds
from src.token_self_repair.evaluation.reasoning_runner import ReasoningBenchmarkResult

# Set style
plt.style.use('default')  # Use 'default' instead of 'seaborn-v0_8' for Colab compatibility
sns.set_palette("husl")

# Configuration
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
USE_QUANTIZATION = True  # Set to False if you have enough GPU memory
THRESHOLD_VALUES = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]
DATASETS = ["gsm8k", "humaneval", "truthfulqa", "bioasq"]  # Add more as needed
MAX_SAMPLES_PER_DATASET = 20  # Adjust based on your needs
RESULTS_DIR = Path("results/threshold_analysis")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"Thresholds to test: {THRESHOLD_VALUES}")
print(f"Datasets: {DATASETS}")
print(f"Results will be saved to: {RESULTS_DIR}")


## Authenticate with Hugging Face


In [ ]:
# Login to Hugging Face (required for Llama models)
from huggingface_hub import login

# Option 1: Use your token directly (replace with your token)
# login(token="your_hf_token_here")

# Option 2: Use interactive login
print("Please login to Hugging Face:")
login()

print("✅ Hugging Face authentication complete!")


## Load Model


In [ ]:
print("Loading model...")
print("This may take a few minutes on first run (downloading ~16GB)...")
llm = load_llama(model_name=MODEL_NAME, quantize=USE_QUANTIZATION)
print("✅ Model loaded successfully!")


## Data Structures for Results


In [ ]:
@dataclass
class ThresholdResult:
    """Results for a single threshold value."""
    threshold: float
    dataset: str
    accuracy_before: float
    accuracy_after: float
    accuracy_improvement: float
    repair_trigger_rate: float
    avg_repairs_per_query: float
    repair_success_rate: float
    false_positive_rate: float
    false_negative_rate: float
    auroc: float
    calibration_error: float
    avg_uncertainty: float
    avg_latency: float
    latency_overhead: float
    num_samples: int
    timestamp: str

# Storage
all_results: List[ThresholdResult] = []


## Helper Functions


In [ ]:
def run_baseline_evaluation(dataset_name: str, max_samples: int) -> ReasoningBenchmarkResult:
    """Run evaluation without repair (baseline)."""
    def factory():
        coordinator = default_reasoning_coordinator(llm)
        coordinator.config.thresholds.repair_activation_uncertainty = 1.0
        coordinator.config.max_self_repairs = 0
        return coordinator
    runner = ReasoningEvaluationRunner(coordinator_factory=factory)
    return runner.run(dataset_name, max_samples=max_samples)

def run_threshold_evaluation(dataset_name: str, threshold: float, max_samples: int) -> ReasoningBenchmarkResult:
    """Run evaluation with a specific threshold."""
    def factory():
        coordinator = default_reasoning_coordinator(llm)
        coordinator.config.thresholds.repair_activation_uncertainty = threshold
        coordinator.config.max_self_repairs = 3
        return coordinator
    runner = ReasoningEvaluationRunner(coordinator_factory=factory)
    return runner.run(dataset_name, max_samples=max_samples)

def calculate_repair_metrics(baseline_result: ReasoningBenchmarkResult, 
                            repair_result: ReasoningBenchmarkResult) -> Dict:
    """Calculate repair-specific metrics."""
    repairs_triggered = sum(1 for s in repair_result.samples if s.final_uncertainty > 0.3)
    total_repairs = repairs_triggered
    successful_repairs = 0
    false_positives = 0
    false_negatives = 0
    
    baseline_dict = {s.prompt: s for s in baseline_result.samples}
    for repair_sample in repair_result.samples:
        baseline_sample = baseline_dict.get(repair_sample.prompt)
        if not baseline_sample:
            continue
        triggered = repair_sample.final_uncertainty > 0.3
        if triggered:
            if not baseline_sample.correct and repair_sample.correct:
                successful_repairs += 1
            elif baseline_sample.correct and not repair_sample.correct:
                false_positives += 1
        else:
            if not baseline_sample.correct and not repair_sample.correct:
                false_negatives += 1
    
    num_samples = len(repair_result.samples)
    return {
        "repair_trigger_rate": repairs_triggered / num_samples if num_samples > 0 else 0.0,
        "avg_repairs_per_query": total_repairs / num_samples if num_samples > 0 else 0.0,
        "repair_success_rate": successful_repairs / repairs_triggered if repairs_triggered > 0 else 0.0,
        "false_positive_rate": false_positives / repairs_triggered if repairs_triggered > 0 else 0.0,
        "false_negative_rate": false_negatives / (num_samples - repairs_triggered) if (num_samples - repairs_triggered) > 0 else 0.0,
    }


In [ ]:
print("Starting threshold sweep evaluation...")
print(f"This will test {len(THRESHOLD_VALUES)} thresholds across {len(DATASETS)} datasets")
print(f"Total evaluations: {len(THRESHOLD_VALUES) * len(DATASETS) * 2} (baseline + repair for each)")
print("\nThis may take a while. Progress will be shown below.\n")

baseline_results = {}

# First, run baseline evaluations (no repair)
print("\n=== Running Baseline Evaluations (No Repair) ===")
for dataset in DATASETS:
    print(f"\nDataset: {dataset}")
    try:
        baseline = run_baseline_evaluation(dataset, MAX_SAMPLES_PER_DATASET)
        baseline_results[dataset] = baseline
        print(f"  Baseline Accuracy: {baseline.accuracy:.3f}")
    except Exception as e:
        print(f"  ❌ Error: {e}")
        baseline_results[dataset] = None

# Now run threshold sweep
print("\n\n=== Running Threshold Sweep ===")
for threshold in tqdm(THRESHOLD_VALUES, desc="Thresholds"):
    for dataset in DATASETS:
        if baseline_results.get(dataset) is None:
            continue
        try:
            repair_result = run_threshold_evaluation(dataset, threshold, MAX_SAMPLES_PER_DATASET)
            baseline = baseline_results[dataset]
            repair_metrics = calculate_repair_metrics(baseline, repair_result)
            latency_overhead = repair_result.average_latency - baseline.average_latency
            
            result = ThresholdResult(
                threshold=threshold, dataset=dataset,
                accuracy_before=baseline.accuracy, accuracy_after=repair_result.accuracy,
                accuracy_improvement=repair_result.accuracy - baseline.accuracy,
                repair_trigger_rate=repair_metrics["repair_trigger_rate"],
                avg_repairs_per_query=repair_metrics["avg_repairs_per_query"],
                repair_success_rate=repair_metrics["repair_success_rate"],
                false_positive_rate=repair_metrics["false_positive_rate"],
                false_negative_rate=repair_metrics["false_negative_rate"],
                auroc=repair_result.auroc, calibration_error=repair_result.calibration_error,
                avg_uncertainty=repair_result.average_uncertainty,
                avg_latency=repair_result.average_latency, latency_overhead=latency_overhead,
                num_samples=len(repair_result.samples), timestamp=datetime.now().isoformat(),
            )
            all_results.append(result)
        except Exception as e:
            print(f"\n❌ Error at threshold={threshold}, dataset={dataset}: {e}")
            continue

print("\n✅ Evaluation complete!")


In [ ]:
# Save results
df_results = pd.DataFrame([asdict(r) for r in all_results])
csv_path = RESULTS_DIR / f"threshold_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_results.to_csv(csv_path, index=False)
print(f"Results saved to: {csv_path}")

# Download from Colab (if running in Colab)
if 'google.colab' in str(get_ipython()):
    from google.colab import files
    files.download(str(csv_path))
    print("✅ File downloaded to your computer!")

print(f"\nTotal evaluations: {len(all_results)}")
if len(df_results) > 0:
    print(f"Datasets: {df_results['dataset'].unique()}")
    print(f"Thresholds tested: {sorted(df_results['threshold'].unique())}")


## Quick Visualization

For full visualizations, copy the visualization cells from the original notebook.


In [ ]:
if len(df_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Threshold Analysis: Accuracy Metrics', fontsize=16, fontweight='bold')
    
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset].sort_values('threshold')
        axes[0, 0].plot(subset['threshold'], subset['accuracy_before'], 'o--', label=f'{dataset} (before)', alpha=0.7)
        axes[0, 0].plot(subset['threshold'], subset['accuracy_after'], 'o-', label=f'{dataset} (after)', linewidth=2)
        axes[0, 1].plot(subset['threshold'], subset['accuracy_improvement'], 'o-', label=dataset, linewidth=2)
        axes[1, 0].plot(subset['threshold'], subset['repair_trigger_rate'] * 100, 'o-', label=dataset, linewidth=2)
        axes[1, 1].plot(subset['threshold'], subset['repair_success_rate'] * 100, 'o-', label=dataset, linewidth=2)
    
    axes[0, 0].set_xlabel('Threshold'); axes[0, 0].set_ylabel('Accuracy'); axes[0, 0].set_title('Accuracy Before vs After')
    axes[0, 1].set_xlabel('Threshold'); axes[0, 1].set_ylabel('Accuracy Improvement'); axes[0, 1].set_title('Accuracy Improvement')
    axes[1, 0].set_xlabel('Threshold'); axes[1, 0].set_ylabel('Repair Trigger Rate (%)'); axes[1, 0].set_title('Repair Trigger Rate')
    axes[1, 1].set_xlabel('Threshold'); axes[1, 1].set_ylabel('Repair Success Rate (%)'); axes[1, 1].set_title('Repair Success Rate')
    
    for ax in axes.flat:
        ax.legend(); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Find optimal threshold
    print("\n=== Optimal Thresholds ===")
    for dataset in df_results['dataset'].unique():
        subset = df_results[df_results['dataset'] == dataset]
        best = subset.loc[subset['accuracy_improvement'].idxmax()]
        print(f"{dataset}: {best['threshold']:.3f} (improvement: {best['accuracy_improvement']:+.3f})")
else:
    print("No results to plot. Run the evaluation loop first.")
